### Dependencies

In [37]:
import pandas as pd 
import numpy as np
import gspread as gs
from gspread_dataframe import set_with_dataframe
from gspread_formatting import CellFormat, Color, set_frozen, set_column_width, format_cell_range

### Data overview

In [38]:
# prime_dt = pd.read_csv("./LayerSC/Layer10.csv")
# print(prime_dt,'\n')

# mean = prime_dt.mean()  
# sum = prime_dt.sum()   
# std_dev = prime_dt.loc[:, prime_dt.columns != 'W_content'].std(ddof=1)
# variance = prime_dt.loc[:, prime_dt.columns != 'W_content'].var(ddof=1)

# # print(prime_dt.columns)
# print('Standard Deviation:--------------\n',np.round(std_dev, 4), '\n')
# print('Variance:--------------\n',np.round(variance, 4), '\n')
# print('Mean:--------------\n',np.round(mean, 2), '\n')
# print('Sum:--------------\n',sum, '\n')

## Calculate limit state design values

In [49]:
def soil_limitstate_value(dt_count, dt_variance, characteristic_value, t_path):
    """
    This function calculates the ultimate and serviceability limit state design values for soil properties.

    Parameters:
        dt_count (int): The count of data points in the dataset.
        dt_variance (float): The variance of the dataset.
        characteristic_value (float): The characteristic value of the dataset.
        t_path (str): The file path to the CSV file containing t-coefficients.

    Returns:
        tuple: A tuple containing the ultimate limit state design value (rho_ultimate) and the serviceability limit state design value (rho_serviceability).
    """
    
    ## limit state design value:
    t_coef_data = pd.read_csv(t_path)

    t_1 = t_coef_data[t_coef_data['n'] == 40]['0.95'].values[0] ##todo 
    t_2 = t_coef_data[t_coef_data['n'] == 40]['0.85'].values[0] ##todo
    n_index = dt_count - 1

    print('Design values:')
    print('Ultimate limit state design value t (TTGH I): ',t_1)
    print('Serviceability limit state design value t (TTGH II): ',t_2)

    rho_ultimate =  (t_1 - dt_variance) / np.sqrt(n_index)
    rho_serviceability = (t_2 - dt_variance) / np.sqrt(n_index)

    print('rho_ultimate:', rho_ultimate)
    print('rho_serviceability:', rho_serviceability)
    print('---------------------------------------------')

    print(f'gamma_I = {characteristic_value:.2f}(1 ± {rho_ultimate:.4f})')
    print(f'gamma_II = {characteristic_value:.2f}(1 ± {rho_serviceability:.4f})')

    return rho_ultimate, rho_serviceability

In [46]:
def DesignValuesSheetUpdate(authpath: str, clean_data, characteristic_value, rho_ultimate, rho_serviceability):
    
    gc = gs.service_account(filename=authpath)

    ouputcell_format = CellFormat(
        backgroundColor=Color.fromHex('#80b781'),  # Green color
        textFormat={'italic': True, 'fontSize': 14, 'fontFamily': 'Montserrat'}
    )

    workingSheet = gc.open('Pysheet').worksheet('worksheet') 
    # Update the sheet with the clean_data dataframe
    set_with_dataframe(workingSheet, clean_data)

    format_cell_range(workingSheet, 'E4:E5', ouputcell_format)
    workingSheet.update('E4:E5', [['gamma I'], ['gamma II']]) 
    workingSheet.update('F4:F5', [[f'{characteristic_value:.2f}(1 ± {rho_ultimate:.4f})'], [f'{characteristic_value:.2f}(1 ± {rho_serviceability:.4f})']])



In [55]:
def soilProps_Stat (path):
    """
    This function calculates various statistical properties of soil samples from a given CSV file path.
    It reads the data, calculates variance, mean, and standard deviation, and filters the data based on a condition.
    It also prints the characteristic value and other statistical properties.

    Parameters:
        path (str): The file path to the CSV file containing soil sample data.

    Returns:
        tuple: A tuple containing the cleaned data, characteristic value, ultimate limit state design value (rho_U), and serviceability limit state design value
        (rho_S).
    """
    prime_dt = pd.read_csv(path)

    Soil_Prop_dt = prime_dt['Wet_U_weight']
    Soil_Prop_dt.index.name = None
    Soil_Prop_dt = Soil_Prop_dt.to_frame()

    Soil_Prop_var = Soil_Prop_dt['Wet_U_weight'].var()
    print('OK ✅' if Soil_Prop_var < 0.05 else 'Failed ❌')

    Soil_Prop_count = Soil_Prop_dt['Wet_U_weight'].count()
    print('count:',Soil_Prop_count)

    Soil_Prop_mean = Soil_Prop_dt['Wet_U_weight'].mean()
    print('mean:',Soil_Prop_mean)

    Sample_pass_cond = Soil_Prop_dt['Wet_U_weight'].std()
    print('[v]=', Sample_pass_cond) 

    Soil_Prop_dt['Ad'] = (Soil_Prop_dt['Wet_U_weight'] - Soil_Prop_mean).abs()

    Soil_Prop_dt['Check'] = Soil_Prop_dt['Ad'] < Sample_pass_cond

    clean_data = Soil_Prop_dt.loc[Soil_Prop_dt['Check'] == True]
    clean_data.Name = 'Clean_data'
    characteristic_value = Soil_Prop_dt['Wet_U_weight'].mean()  ##todo  

    print('characteristic value:', np.round(characteristic_value, 2))
    print('---------------------------------------------')

    rho_U, rho_S = soil_limitstate_value(Soil_Prop_count, Soil_Prop_var, characteristic_value, "./Coefficients/t_coef_sheet.csv")

    return clean_data, characteristic_value, rho_U, rho_S 

clean_data, characteristic_value, rho_U, rho_S = soilProps_Stat("./LayerSC/Layer4.csv")
DesignValuesSheetUpdate('pysheetAuth.json', clean_data, characteristic_value, rho_U, rho_S)

OK ✅
count: 10
mean: 1.924
[v]= 0.012649110640673528
characteristic value: 1.92
---------------------------------------------
Design values:
Ultimate limit state design value t (TTGH I):  1.68
Serviceability limit state design value t (TTGH II):  1.05
rho_ultimate: 0.5599466666666667
rho_serviceability: 0.3499466666666667
---------------------------------------------
gamma_I = 1.92(1 ± 0.5599)
gamma_II = 1.92(1 ± 0.3499)


C:\Users\PC\AppData\Local\Temp\ipykernel_21788\1641183003.py:15: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  workingSheet.update('E4:E5', [['gamma I'], ['gamma II']])
C:\Users\PC\AppData\Local\Temp\ipykernel_21788\1641183003.py:16: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  workingSheet.update('F4:F5', [[f'{characteristic_value:.2f}(1 ± {rho_ultimate:.4f})'], [f'{characteristic_value:.2f}(1 ± {rho_serviceability:.4f})']])


### Optional for Google Sheets
this code block is used for converting dataframe to a spreadsheet for a Google sheet file.

### REFACTORING section

In [60]:
from utils import read_csv_files    

folder_path = "./LayerSC"  # Replace with the actual path to your folder
dataframes = read_csv_files(folder_path)

if dataframes:
    for filename, df in dataframes.items():
        print(f"🌟 Calculation for {filename}")
        soilProps_Stat(f'{folder_path}/{filename}') 
        print(f'{'=' * 150} \n \n')

🌟 Calculation for Layer1.csv
OK ✅
count: 14
mean: 1.5021428571428574
[v]= 0.05521685520654997
characteristic value: 1.5
---------------------------------------------
Design values:
Ultimate limit state design value t (TTGH I):  1.68
Serviceability limit state design value t (TTGH II):  1.05
rho_ultimate: 0.46510255181027654
rho_serviceability: 0.2903719899993294
---------------------------------------------
gamma_I = 1.50(1 ± 0.4651)
gamma_II = 1.50(1 ± 0.2904)
 

🌟 Calculation for Layer10.csv
OK ✅
count: 40
mean: 1.97025
[v]= 0.024017888632412596
characteristic value: 1.97
---------------------------------------------
Design values:
Ultimate limit state design value t (TTGH I):  1.68
Serviceability limit state design value t (TTGH II):  1.05
rho_ultimate: 0.2689229270299764
rho_serviceability: 0.16804219013277152
---------------------------------------------
gamma_I = 1.97(1 ± 0.2689)
gamma_II = 1.97(1 ± 0.1680)
 

🌟 Calculation for Layer4.csv
OK ✅
count: 10
mean: 1.924
[v]= 0.0126491